In [ ]:
!pip -q install autogluon

In [35]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.signal import welch
from tqdm import tqdm
from autogluon.tabular import TabularDataset, TabularPredictor

In [ ]:
!gdown 1HqsyO02tVwZINyEp6bZNnTy8v7uuMhK5

In [ ]:
!unzip /content/super-ai-engineer-ss-6-sleep-stage-classification.zip

In [ ]:
def process_single_file_for_ml(file_path, window_size=480):
    df = pd.read_csv(file_path)
    features_list = []

    for i in range(0, len(df) - window_size + 1, window_size):
        window = df.iloc[i : i + window_size]
        f = {}

        f['Sleep_Stage'] = window['Sleep_Stage'].iloc[-1]

        acc_mag = np.sqrt(window['ACC_X']**2 + window['ACC_Y']**2 + window['ACC_Z']**2)
        f['acc_mag_mean'] = acc_mag.mean()
        f['acc_mag_std'] = acc_mag.std()

        freqs, psd = welch(window['BVP'].values, fs=16)
        f['bvp_psd_mean'] = np.mean(psd)
        f['bvp_psd_max'] = np.max(psd)

        freqs_acc, psd_acc = welch(acc_mag.values, fs=16)
        f['acc_energy_low'] = np.sum(psd_acc[freqs_acc < 3])

        # IBI (HRV)
        ibi_vals = window['IBI'].replace(0, np.nan).dropna().values
        if len(ibi_vals) > 1:
            f['ibi_rmssd'] = np.sqrt(np.mean(np.diff(ibi_vals)**2))
            f['ibi_sdnn'] = np.std(ibi_vals)
        else:
            f['ibi_rmssd'], f['ibi_sdnn'] = 0, 0

        # สัญญาณอื่นๆ
        f['hr_mean'] = window['HR'].mean()
        f['eda_mean'] = window['EDA'].mean()
        f['eda_trend'] = window['EDA'].iloc[-1] - window['EDA'].iloc[0]
        f['temp_mean'] = window['TEMP'].mean()
        f['temp_trend'] = window['TEMP'].iloc[-1] - window['TEMP'].iloc[0]
        f['bvp_std'] = window['BVP'].std()

        features_list.append(f)

    df_feat = pd.DataFrame(features_list)

    cols_to_shift = ['hr_mean', 'acc_mag_mean', 'temp_mean', 'eda_mean', 'ibi_rmssd']
    for col in cols_to_shift:
        df_feat[f'{col}_lag1'] = df_feat[col].shift(1)
        df_feat[f'{col}_lag2'] = df_feat[col].shift(2)
        df_feat[f'{col}_lag3'] = df_feat[col].shift(3)
        df_feat[f'{col}_diff1'] = df_feat[col] - df_feat[col].shift(1)

    for col in ['temp_mean', 'eda_mean', 'hr_mean', 'acc_mag_mean']:
        df_feat[f'{col}_roll_mean_10'] = df_feat[col].rolling(window=10, min_periods=1).mean()
        df_feat[f'{col}_roll_std_10'] = df_feat[col].rolling(window=10, min_periods=1).std()

    df_feat = df_feat.bfill()

    return df_feat

In [ ]:
def process_test_folder(folder_path):
    """
    ฟังก์ชันนี้จะดึงไฟล์ .csv ย่อยๆ ทั้งหมดของคน 1 คน (เช่นโฟลเดอร์ test001)
    มาเรียงตามเวลา สกัดฟีเจอร์ และทำ Contextual Features ให้ถูกต้อง
    """
    files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))
    features_list = []

    for f_path in files:
        file_id = os.path.basename(f_path).replace('.csv', '')
        window = pd.read_csv(f_path)

        f = {'id': file_id}

        acc_mag = np.sqrt(window['ACC_X']**2 + window['ACC_Y']**2 + window['ACC_Z']**2)
        f['acc_mag_mean'] = acc_mag.mean()
        f['acc_mag_std'] = acc_mag.std()

        freqs, psd = welch(window['BVP'].values, fs=16)
        f['bvp_psd_mean'] = np.mean(psd)
        f['bvp_psd_max'] = np.max(psd)

        freqs_acc, psd_acc = welch(acc_mag.values, fs=16)
        f['acc_energy_low'] = np.sum(psd_acc[freqs_acc < 3])

        ibi_vals = window['IBI'].replace(0, np.nan).dropna().values
        if len(ibi_vals) > 1:
            f['ibi_rmssd'] = np.sqrt(np.mean(np.diff(ibi_vals)**2))
            f['ibi_sdnn'] = np.std(ibi_vals)
        else:
            f['ibi_rmssd'], f['ibi_sdnn'] = 0, 0

        f['hr_mean'] = window['HR'].mean()
        f['eda_mean'] = window['EDA'].mean()
        f['eda_trend'] = window['EDA'].iloc[-1] - window['EDA'].iloc[0]
        f['temp_mean'] = window['TEMP'].mean()
        f['temp_trend'] = window['TEMP'].iloc[-1] - window['TEMP'].iloc[0]
        f['bvp_std'] = window['BVP'].std()

        features_list.append(f)

    df_feat = pd.DataFrame(features_list)

    cols_to_shift = ['hr_mean', 'acc_mag_mean', 'temp_mean', 'eda_mean', 'ibi_rmssd']
    for col in cols_to_shift:
        df_feat[f'{col}_lag1'] = df_feat[col].shift(1)
        df_feat[f'{col}_lag2'] = df_feat[col].shift(2)
        df_feat[f'{col}_lag3'] = df_feat[col].shift(3)
        df_feat[f'{col}_diff1'] = df_feat[col] - df_feat[col].shift(1)

    for col in ['temp_mean', 'eda_mean', 'hr_mean', 'acc_mag_mean']:
        df_feat[f'{col}_roll_mean_10'] = df_feat[col].rolling(window=10, min_periods=1).mean()
        df_feat[f'{col}_roll_std_10'] = df_feat[col].rolling(window=10, min_periods=1).std()

    df_feat = df_feat.bfill()
    return df_feat

In [ ]:
data_folder = '/content/train/train'
all_files = glob.glob(os.path.join(data_folder, "*.csv"))

print(f"พบไฟล์ทั้งหมด {len(all_files)} ไฟล์ เริ่มทำการสกัดฟีเจอร์...")

all_features_dfs = []

for file in tqdm(all_files, desc="Extracting Features"):
    try:
        df_person_features = process_single_file_for_ml(file)
        all_features_dfs.append(df_person_features)
    except Exception as e:
        print(f"\nข้ามไฟล์ {file} เกิดข้อผิดพลาด: {e}")

final_ml_dataset = pd.concat(all_features_dfs, ignore_index=True)

print(f"\n✅ แปลงข้อมูลเสร็จสิ้น! ได้ข้อมูลทั้งหมด {final_ml_dataset.shape[0]} แถว (Windows)")
print(f"✅ จำนวน Features ที่จะใช้เทรน: {final_ml_dataset.shape[1] - 1} ฟีเจอร์")

พบไฟล์ทั้งหมด 83 ไฟล์ เริ่มทำการสกัดฟีเจอร์...


Extracting Features: 100%|██████████| 83/83 [04:03<00:00,  2.93s/it]


✅ แปลงข้อมูลเสร็จสิ้น! ได้ข้อมูลทั้งหมด 66745 แถว (Windows)
✅ จำนวน Features ที่จะใช้เทรน: 41 ฟีเจอร์


In [ ]:
test_folder = '/content/test_segment/test_segment/'

test_folders = [os.path.join(test_folder, f"test{i:03d}") for i in range(1, 11)]

all_test_features_dfs = []
for folder in tqdm(test_folders, desc="Extracting Test Features"):
    if os.path.exists(folder):
        df_person_test = process_test_folder(folder)
        all_test_features_dfs.append(df_person_test)

final_ml_test_dataset = pd.concat(all_test_features_dfs, ignore_index=True)
print(f"Test Data พร้อม! มีทั้งหมด {final_ml_test_dataset.shape[0]} แถว")

Extracting Test Features: 100%|██████████| 10/10 [00:41<00:00,  4.15s/it]

Test Data พร้อม! มีทั้งหมด 7832 แถว


In [ ]:
print(final_ml_dataset['Sleep_Stage'].unique())

final_ml_dataset['Sleep_Stage'] = final_ml_dataset['Sleep_Stage'].astype(str).str.strip().str.upper()

mapping = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
final_ml_dataset['Sleep_Stage'] = final_ml_dataset['Sleep_Stage'].map(mapping)

print(final_ml_dataset['Sleep_Stage'].isnull().sum())

['W' 'N2' 'N1' 'N3' 'R']
0


In [ ]:
save_path = 'best_model'
hyperparameters = {
    'GBM': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'CAT': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'XGB': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'RF': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],

}
time_limit=7200

In [ ]:
class_counts = final_ml_dataset['Sleep_Stage'].value_counts()
total_samples = len(final_ml_dataset)
weights = {cls: total_samples / count for cls, count in class_counts.items()}

final_ml_dataset['sample_weight'] = final_ml_dataset['Sleep_Stage'].map(weights)

predictor = TabularPredictor(
                            label='Sleep_Stage',
                            sample_weight='sample_weight',
                            problem_type='multiclass',
                            path=save_path,
                            eval_metric='f1_weighted'
                            ).fit(
                                final_ml_dataset,
                                  presets='high_quality',
                                  time_limit=time_limit,
                                  num_bag_folds=5,
                                  num_bag_sets=1,
                                  num_stack_levels=1,
                                  auto_stack=True
                            )
print("Finish!")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       9.15 GB / 12.67 GB (72.2%)
Disk Space Avail:   57.34 GB / 112.64 GB (50.9%)
Presets specified: ['high_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~5x), but runs the risk of an out-of

Finish!


In [42]:
y_pred = predictor.predict_proba(final_ml_test_dataset)
y_pred.head()

,0,1,2,3,4
0,0.766040,0.085183,0.137699,0.005419,0.005660
1,0.687256,0.073212,0.231759,0.006280,0.001492
2,0.422953,0.068799,0.476939,0.028556,0.002753
3,0.530243,0.058099,0.396397,0.013713,0.001548
4,0.803812,0.057513,0.131719,0.005300,0.001656


In [ ]:
inv_mapping = {0: 'W', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'R'}
final_ml_test_dataset['labels'] = y_pred.max()

y_pred_class = y_pred.idxmax(axis=1)

inv_mapping = {0: 'W', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'R'}
final_ml_test_dataset['labels'] = y_pred_class.map(inv_mapping)

submission_df = final_ml_test_dataset[['id', 'labels']]
submission_df.to_csv('submission_autogluon_proba.csv', index=False)

print("ดึงคลาสที่มั่นใจที่สุด และสร้างไฟล์ส่งสำเร็จ!")
print(submission_df.head())

ดึงคลาสที่มั่นใจที่สุด และสร้างไฟล์ส่งสำเร็จ!
              id labels
0  test001_00000      W
1  test001_00001      W
2  test001_00002     N2
3  test001_00003      W
4  test001_00004      W


In [45]:
from google.colab import files

files.download('submission_autogluon_proba.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>